# Preprocesamiento de datos y feature engineering

In [1]:
from ucimlrepo import fetch_ucirepo 
import pandas as pd

bank_marketing = fetch_ucirepo(id=222)

X = bank_marketing.data.features 
y = bank_marketing.data.targets

df_work = pd.concat([X, y], axis=1)

df_work.head(5)

,age,job,marital,education,default,balance,housing,loan,contact,day_of_week,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,NaN,5,may,261,1,-1,0,NaN,no
1,44,technician,single,secondary,no,29,yes,no,NaN,5,may,151,1,-1,0,NaN,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,NaN,5,may,76,1,-1,0,NaN,no
3,47,blue-collar,married,NaN,no,1506,yes,no,NaN,5,may,92,1,-1,0,NaN,no
4,33,NaN,single,NaN,no,1,no,no,NaN,5,may,198,1,-1,0,NaN,no


## Para las variables con valores nulos se crean nuevas columnas donde se distingue si el valor es desconocido o no

In [2]:
df_work['job_unknown'] = (df_work['job'] == 'unknown').astype(int)
df_work['education_unknown'] = (df_work['education'] == 'unknown').astype(int)
df_work['contact_unknown'] = (df_work['contact'] == 'unknown').astype(int)
df_work['poutcome_unknown'] = (df_work['poutcome'] == 'unknown').astype(int)


## Se tratan los outliers

In [3]:
def cap_outliers(df, column, lower_percentile=1, upper_percentile=99):
    """Limita valores extremos a percentiles"""
    lower = df[column].quantile(lower_percentile/100)
    upper = df[column].quantile(upper_percentile/100)
    df[column] = df[column].clip(lower=lower, upper=upper)
    return df

df_work = cap_outliers(df_work, 'balance', 1, 99)
df_work = cap_outliers(df_work, 'duration', 1, 99)
df_work = cap_outliers(df_work, 'campaign', 1, 99)

print("Outliers tratados con capping (percentiles 1-99)")

Outliers tratados con capping (percentiles 1-99)


## Se crean nuevas variable

In [4]:
df_work['is_contacted_before'] = (df_work['pdays'] != -1).astype(int)
df_work['previous_exit'] = (df_work['poutcome'] == 'success').astype(int)
df_work['positive_balance'] = (df_work['balance'] > 0).astype(int)


In [5]:
df_work['duration']

0         261
1         151
2          76
3          92
4         198
         ... 
45206     977
45207     456
45208    1127
45209     508
45210     361
Name: duration, Length: 45211, dtype: int64

## Conversion de la variable target a binario numerico

In [6]:
df_work['y'] = (df_work['y'] == 'yes').astype(int)



## Se realiza one hot encoding a variables categoricas

In [7]:
categorical_cols = ['job', 'marital', 'education', 'default', 'housing', 
                   'loan', 'contact', 'month', 'day_of_week', 'poutcome']

df_work = pd.get_dummies(df_work, columns=categorical_cols, drop_first=False)

print(f"Dimensiones finales: {df_work.shape}")
df_work.columns

Dimensiones finales: (45211, 85)


Index(['age', 'balance', 'duration', 'campaign', 'pdays', 'previous', 'y',
       'job_unknown', 'education_unknown', 'contact_unknown',
       'poutcome_unknown', 'is_contacted_before', 'previous_exit',
       'positive_balance', 'job_admin.', 'job_blue-collar', 'job_entrepreneur',
       'job_housemaid', 'job_management', 'job_retired', 'job_self-employed',
       'job_services', 'job_student', 'job_technician', 'job_unemployed',
       'marital_divorced', 'marital_married', 'marital_single',
       'education_primary', 'education_secondary', 'education_tertiary',
       'default_no', 'default_yes', 'housing_no', 'housing_yes', 'loan_no',
       'loan_yes', 'contact_cellular', 'contact_telephone', 'month_apr',
       'month_aug', 'month_dec', 'month_feb', 'month_jan', 'month_jul',
       'month_jun', 'month_mar', 'month_may', 'month_nov', 'month_oct',
       'month_sep', 'day_of_week_1', 'day_of_week_2', 'day_of_week_3',
       'day_of_week_4', 'day_of_week_5', 'day_of_week_6', 'day_

## Guardar el dataframe

In [8]:
df_work.to_csv('../data/clean/bank_marketing_clean.csv', index=False)